In [14]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import HistGradientBoostingClassifier
table = pd.read_parquet("../data/processed/model_table.parquet")
table = table[table["cik"] != "0000000000"]          # sécurité

CIBLES   = ["y_A", "y_L", "y_AC", "y_LC", "y_C", "y_E"]
FEATURES = [c for c in table if c not in ("cik", "end") and not c.startswith("y_")]

annee = table["end"].dt.year
table["split"] = np.select(
    [annee <= 2019, annee <= 2021, annee <= 2024],
    ["train", "val", "test"],
    default="prediction",          # 2025-2026 : pas encore de t+1
)
table.groupby("split").agg(bilans=("cik", "size"),
                           entreprises=("cik", "nunique"),
                           avec_cible_y_A=("y_A", lambda s: s.notna().sum()))

,bilans,entreprises,avec_cible_y_A
split,,,
prediction,5771,5148,617
test,16954,6549,15186
train,58289,9432,52938
val,11817,6574,10739


In [15]:
train = table[table["split"] == "train"]
bornes = {k: tuple(train[k].quantile([0.01, 0.99])) for k in CIBLES}
pd.DataFrame(bornes, index=["p1", "p99"]).round(3)

,y_A,y_L,y_AC,y_LC,y_C,y_E
p1,-0.958,-1.531,-1.486,-1.565,-0.482,-0.991
p99,1.658,2.162,2.231,2.173,1.777,4.410


In [16]:
q = train[FEATURES].quantile([0.01, 0.99])
def clip_X(df):
    return df[FEATURES].clip(q.loc[0.01], q.loc[0.99], axis=1)

In [17]:
def scores(y, pred):
    pred = pd.Series(pred, index=y.index)
    sens = y != 0                                         # on ignore les variations exactement nulles
    dir_acc = np.nan if (pred == 0).all() else (np.sign(pred[sens]) == np.sign(y[sens])).mean()
    return {"MAE": mean_absolute_error(y, pred), "dir_acc": dir_acc}

In [18]:
EVAL = "val"            # on ne passera à "test" qu'à la toute fin
resultats = []

for k in CIBLES:
    lo, hi = bornes[k]
    tr = table[(table["split"] == "train") & table[k].notna()]
    ev = table[(table["split"] == EVAL) & table[k].notna()]
    y_tr, y_ev = tr[k].clip(lo, hi), ev[k].clip(lo, hi)
    passe = k.replace("y_", "d_")

    preds = {
        "Zéro (aucun changement)": np.zeros(len(ev)),
        "Médiane du train (≈ toujours hausse)": np.full(len(ev), y_tr.median()),
        "Persistance (= variation passée)": ev[passe].fillna(y_tr.median()).clip(lo, hi),
    }

    # ← NOUVEAU : repère = sens le plus fréquent DANS la période évaluée (pas un vrai modèle)
    sens_majoritaire = np.sign(y_ev[y_ev != 0]).mode()[0]
    preds["Classe majoritaire de la période (repère)"] = np.full(len(ev), sens_majoritaire * 1e-6)

    ridge = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0))
    ridge.fit(clip_X(tr), y_tr)
    preds["Ridge"] = ridge.predict(clip_X(ev))

    hgb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                        min_samples_leaf=100, l2_regularization=1.0,
                                        random_state=0)
    hgb.fit(tr[FEATURES], y_tr)
    preds["HistGradientBoosting"] = hgb.predict(ev[FEATURES])

    # ← NOUVEAU : classifieur entraîné directement sur le SENS (hausse = 1, baisse = 0)
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                         min_samples_leaf=100, l2_regularization=1.0,
                                         random_state=0)
    clf.fit(tr[FEATURES], (y_tr > 0).astype(int))
    proba = clf.predict_proba(ev[FEATURES])[:, 1]
    preds["HGB classifieur (sens)"] = np.where(proba > 0.5, 1e-6, -1e-6)   # ← NOUVEAU

    for nom, p in preds.items():
        resultats.append({"cible": k, "modèle": nom, **scores(y_ev, p)})

res = pd.DataFrame(resultats)
print("Fini :", len(res), "résultats")

Fini : 42 résultats


In [19]:
print("=== Précision directionnelle (plus haut = mieux) ===")
display(res.pivot(index="modèle", columns="cible", values="dir_acc").round(3))

print("=== MAE (plus bas = mieux) ===")
display(res.pivot(index="modèle", columns="cible", values="MAE").round(3))

=== Précision directionnelle (plus haut = mieux) ===


cible,y_A,y_AC,y_C,y_E,y_L,y_LC
modèle,,,,,,
Classe majoritaire de la période (repère),0.628,0.577,0.538,0.581,0.636,0.706
HGB classifieur (sens),0.644,0.635,0.621,0.627,0.658,0.703
HistGradientBoosting,0.647,0.626,0.539,0.585,0.651,0.704
Médiane du train (≈ toujours hausse),0.628,0.577,0.462,0.581,0.636,0.706
Persistance (= variation passée),0.631,0.508,0.445,0.588,0.607,0.582
Ridge,0.628,0.588,0.528,0.539,0.642,0.672
Zéro (aucun changement),NaN,NaN,NaN,NaN,NaN,NaN


=== MAE (plus bas = mieux) ===


cible,y_A,y_AC,y_C,y_E,y_L,y_LC
modèle,,,,,,
Classe majoritaire de la période (repère),0.267,0.453,0.166,0.324,0.346,0.441
HGB classifieur (sens),0.267,0.453,0.166,0.324,0.346,0.441
HistGradientBoosting,0.253,0.396,0.164,0.298,0.318,0.344
Médiane du train (≈ toujours hausse),0.263,0.449,0.166,0.323,0.338,0.418
Persistance (= variation passée),0.379,0.677,0.288,0.551,0.476,0.582
Ridge,0.268,0.445,0.174,0.362,0.339,0.389
Zéro (aucun changement),0.267,0.453,0.166,0.324,0.346,0.441


In [20]:
from sklearn.metrics import roc_auc_score

FEATURES_BILAN = ["taille", "L_A", "E_A", "C_A", "RE_A", "AC_LC", "AC_A", "LC_A",
                  "PPE_A", "incorp_A", "E_negatif",
                  "d_A", "d_L", "d_AC", "d_LC", "d_C", "d_E"]

comparaison = []
for k in CIBLES:
    lo, hi = bornes[k]
    tr = table[(table["split"] == "train") & table[k].notna()]
    ev = table[(table["split"] == "val") & table[k].notna()]
    y_tr, y_ev = tr[k].clip(lo, hi), ev[k].clip(lo, hi)

    for nom, cols in [("Bilan seul (17)", FEATURES_BILAN), ("Bilan + flux (30)", FEATURES)]:
        clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                             min_samples_leaf=100, l2_regularization=1.0,
                                             random_state=0)
        clf.fit(tr[cols], (y_tr > 0).astype(int))
        proba = clf.predict_proba(ev[cols])[:, 1]

        reg = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                            min_samples_leaf=100, l2_regularization=1.0,
                                            random_state=0)
        reg.fit(tr[cols], y_tr)

        comparaison.append({
            "cible": k, "features": nom,
            "dir_acc": scores(y_ev, np.where(proba > 0.5, 1e-6, -1e-6))["dir_acc"],
            "AUC": roc_auc_score((y_ev > 0).astype(int), proba),
            "MAE": mean_absolute_error(y_ev, reg.predict(ev[cols])),
        })

comp = pd.DataFrame(comparaison)
for m in ["dir_acc", "AUC", "MAE"]:
    print(f"=== {m} ===")
    display(comp.pivot(index="features", columns="cible", values=m).round(3))

=== dir_acc ===


cible,y_A,y_AC,y_C,y_E,y_L,y_LC
features,,,,,,
Bilan + flux (30),0.644,0.635,0.621,0.627,0.658,0.703
Bilan seul (17),0.648,0.637,0.621,0.616,0.651,0.712


=== AUC ===


cible,y_A,y_AC,y_C,y_E,y_L,y_LC
features,,,,,,
Bilan + flux (30),0.674,0.676,0.671,0.658,0.648,0.669
Bilan seul (17),0.669,0.676,0.669,0.647,0.636,0.669


=== MAE ===


cible,y_A,y_AC,y_C,y_E,y_L,y_LC
features,,,,,,
Bilan + flux (30),0.253,0.396,0.164,0.298,0.318,0.344
Bilan seul (17),0.257,0.396,0.162,0.295,0.320,0.346
